In [1]:
#Importacion de librerias
import requests
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
from googleapiclient.discovery import build
from google.oauth2 import service_account


periodos = ['26ES5', '2026D', '26V05']
periodo_virtual = '26V05'
periodo_presencial = '2026D'
periodo_especializacion = '26ES5'
periodo_pago = '202609'
lider = 'LUZ'

In [2]:
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [3]:
novedades = pd.read_excel('NOVEDADES_BOGOTA_202609.xlsx')
novedades

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,Propietario del registro,Modalidad,Correo Ejecutivo,Documento Ejecutivo,Regional,PERIODO_PAGO,MOTIVO,Unnamed: 16
0,1065615820,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN REGISTRO DE PROCESOS PR...,1-ACTIVO,Vinculaciones Cristian Hernando Pulido Aguilar,Virtual,CRISTIAN_PULIDOA@CUN.EDU.CO,1033736941,Bogotá,202609,NO REGISTRA PAGO,SI TIENE MARCA DE PAGO
1,1022939637,26ES5,PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACION EN CONTRATACION ESTATAL,NaN,Vinculaciones Cristian Hernando Pulido Aguilar,Virtual,CRISTIAN_PULIDOA@CUN.EDU.CO,1033736941,Bogotá,202609,NO CUENTA CON MARCA DE PAGO,SI TIENE MARCA DE PAGO NO ERA VIRTUAL SINO ESP...
2,1147484701,2026D,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,ADMINISTRACION DEPORTIVA,NaN,Vinculaciones Dairon Alejandro Ibañez Rey,Virtual,DAIRON_IBANEZR@CUN.EDU.CO,1019115249,Bogotá,202609,NO CUENTA CON MARCA DE PAGO,SI TIENE MARCA DE PAGO
3,1010211957,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PRODUCCIÓN DE MATERIAS ...,1-ACTIVO,Vinculaciones Dairon Alejandro Ibañez Rey,Virtual,DAIRON_IBANEZR@CUN.EDU.CO,1019115249,Bogotá,202609,NO REGISTRA PAGO,SI TIENE MARCA DE PAGO
4,1089606238,26T05,NaN,VINCULACIONES,NUEVO,NUEVO,Pregrado,BACHILLERSITARIO,1-ACTIVO,Vinculaciones Cristian Hernando Pulido Aguilar,Virtual,CRISTIAN_PULIDOA@CUN.EDU.CO,1033736941,Bogotá,202609,SI TIENE MARCA DE PAGO,SI TIENE MARCA DE PAGO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316,1075238627,26ES5,PAGO,VINCULACIONES,NUEVO,NUEVO,Postgrado,ESPECIALIZACIÓN EN CIBERSEGURIDAD,1-ACTIVO,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,1012382480,Bogotá,202609,NO ESTA OK EN ZOHO,Todo ok
317,1001349100,26V05,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,NaN,1-ACTIVO,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,1012382480,Bogotá,202609,NO ESTA OK EN ZOHO,Todo ok
318,1069985796,26V05,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,ADMINISTRACIÓN DE LA SEGURIDAD Y SALUD EN EL T...,NaN,Vinculaciones Daniel Felipe Prieto Alvarado,Virtual,DANIEL_PRIETO@CUN.EDU.CO,1007101612,Bogotá,202609,NO CUENTA CON MARCA DE PAGO,Si tiene marca de pago.
319,1003950424,26V05,PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,INGENIERIA DE SISTEMAS,NaN,Vinculaciones Daniel Felipe Prieto Alvarado,Virtual,DANIEL_PRIETO@CUN.EDU.CO,1007101612,Bogotá,202609,NO CUENTA CON MARCA DE PAGO,Si tiene marca de pago.


In [4]:
# Autenticación con la clave de servicio
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']
KEY = 'key2.json'  # LLAVES
SPREADSHEET_ID = "15NkOzO6MBlMJjtabMTB1Bc_PEKJ6lb6Rh8won1i1xm0"  # ID DE LA HOJA DE CALCULO

# Crear credenciales
creds = service_account.Credentials.from_service_account_file(KEY, scopes=SCOPES)
service = build('sheets', 'v4', credentials=creds)
sheet = service.spreadsheets()

# Llamar a la API de Google Sheets
RANGE_NAME = "'VENTAS'!A1:AE100000000"  # HOJA Y RANGO A TRABAJAR

result = sheet.values().get(spreadsheetId=SPREADSHEET_ID, range=RANGE_NAME).execute()
values = result.get('values', [])

# Mostrar resultados
if not values:
    print("No se encontraron datos.")
else:
    print(f"Datos obtenidos: {len(values):,} filas")

Datos obtenidos: 21,856 filas


In [5]:
df_excel = pd.DataFrame(values[1:], columns=values[0])
df_excel.shape

(21855, 31)

SE CRUZA CON EL REGISTRO DE VENTAS

In [6]:
df_excel['Número de Documento de estudiante'] = df_excel['Número de Documento de estudiante'].astype(str)
novedades['DOC_ALUM'] = novedades['DOC_ALUM'].astype(str)
df_pre_ventas = df_excel[df_excel['Número de Documento de estudiante'].isin(novedades['DOC_ALUM'])].copy()
df_pre_ventas.shape

(320, 31)

SE DEJAN LAS VENTAS QUE NO ESTAN EN EL REGISTRO DE VENTAS

In [7]:
df_no_preventas = df_pre_ventas[~df_pre_ventas['Número de Documento de estudiante'].isin(novedades['DOC_ALUM'])].copy()
df_no_preventas.shape

(0, 31)

# SE COMPLEMENTA LA INFORMACION DE LOS ASESORES

In [8]:
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [9]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    #archivo_orig = pd.read_sql("select * from CUN_REPOSITORIO.Comisiones.Com_Telec_Oct_2024_2_yCT", engine2_1)
    df_cedula = pd.read_sql(f'''
                    SELECT IDENTIFICACION AS 'Documento Ejecutivo', BOX_MAIL AS 'Correo Ejecutivo' FROM Hub_Empleados
                               ''', engine2_1)
    df_cedula_2 = df_cedula.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_cedula_2.shape

Conexion a la base de datos realizada


Conexión cerrada


(89788, 2)

In [10]:
df_cedula_2['Documento Ejecutivo'] = df_cedula_2['Documento Ejecutivo'].astype(int).astype(str)
df_cedula_2['Correo Ejecutivo'] = df_cedula_2['Correo Ejecutivo'].str.upper()

In [11]:
df_pre_ventas['Correo Ejecutivo'] = df_pre_ventas['Correo Ejecutivo'].str.upper()

In [12]:
# se unifica con la informacion de registro ventas
df_pre_ventas = df_pre_ventas.merge(df_cedula_2,how='left',left_on='Correo Ejecutivo',right_on='Correo Ejecutivo').copy()
df_pre_ventas.drop_duplicates(inplace=True)

SE TRAEN LAS METAS

In [48]:
df_metas_virtual = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_BOGOTA_202609.xlsx', sheet_name= periodo_virtual)
df_metas_presencial = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_BOGOTA_202609.xlsx', sheet_name= periodo_presencial)
df_metas_especializacion = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\METAS_BOGOTA_202609.xlsx', sheet_name= periodo_especializacion)

In [49]:
df_metas_virtual['GESTOR CORREO'] = df_metas_virtual['GESTOR CORREO'].str.upper()
df_ventas_virtual = df_pre_ventas.copy()
df_ventas_virtual.shape

(320, 32)

In [50]:
# TRATAMIENTO DE COLUMNAS
columnas = ['Hora agregado', 'Numero de recibo','Dirección IP','Correo del estudiante','Teléfono del estudiante',
'Ciudad de residencia','Imagen del Pago Generado','% De Descuento',
'Medio de pago','Fuente de pionero','Campana','Fuente campana','Content','Sitio web', 'Tipo de inscripción',
'ID único','Fecha','Documento Ejecutivo_x','Número de aprobación de pago','Tipo de Financiación', 'Convenio', 'Valor pagado']

df_ventas_virtual.drop(columns=columnas, inplace=True)
df_ventas_virtual['Periodo'] = df_ventas_virtual['Periodo'].astype(str).str.strip()
df_ventas_virtual = df_ventas_virtual.rename(columns={'Documento Ejecutivo_y':'Documento Ejecutivo'})
df_ventas_virtual['Documento Ejecutivo'] = df_ventas_virtual['Documento Ejecutivo'].astype(str)
df_ventas_virtual['Número de Documento de estudiante'] = df_ventas_virtual['Número de Documento de estudiante'].astype(str)
df_ventas_virtual['Llave']=df_ventas_virtual['Número de Documento de estudiante']
df_ventas_virtual.shape

(320, 11)

# SE TRAE LA MARCA DE PAGO

In [51]:
periodos_sql = ",".join(f"'{p}'" for p in periodos)

In [52]:
para_orden = df_pre_ventas['Número de Documento de estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"

In [53]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
        SELECT distinct DOC_ALUM,PERIODO,ESTADO_PAGO,fuerzazoho as FuerzaComercialFinal, NUEVO, CLASE_ACTUAL, TIPO, PROGRAMA, EST_MATRICULADO
        FROM ZOHO.Base_Personas WITH(NOLOCK)
        WHERE PERIODO IN ({periodos_sql})
        and doc_alum in ({valores_unicos_df1_orden})
                        ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_recibos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_recibos.shape

(306, 9)

In [54]:
df_recibos['LLAVE_RECIBOS'] = df_recibos['DOC_ALUM'].astype(str)
df_recibos_virtual = df_recibos.copy()
df_recibos_virtual.shape

(306, 10)

SE CRUZA LOS RECIBOS CON EL REGISTRO DE VENTAS

In [55]:
df_recibos_virtual = df_recibos_virtual.merge(df_ventas_virtual[['Propietario del registro','Modalidad','Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave']],how='inner',left_on='LLAVE_RECIBOS',right_on='Llave')
df_recibos_virtual.drop_duplicates(inplace=True)
df_recibos_virtual.shape

(311, 16)

In [56]:
#tratamiento de columnas
df_recibos_virtual['CLASE_ACTUAL'] = df_recibos_virtual['CLASE_ACTUAL'].astype(str).str.upper()
df_recibos_virtual['FuerzaComercialFinal'] = df_recibos_virtual['FuerzaComercialFinal'].str.upper()
df_recibos_virtual['NUEVO'] = df_recibos_virtual['NUEVO'].astype(str).str.upper()
df_recibos_virtual['ESTADO_PAGO'] = df_recibos_virtual['ESTADO_PAGO'].astype(str).str.upper()
df_recibos_virtual['FuerzaComercialFinal'] = df_recibos_virtual['FuerzaComercialFinal'].str.strip()
df_recibos_virtual['NUEVO'] = df_recibos_virtual['NUEVO'].str.strip()
df_recibos_virtual['CLASE_ACTUAL'] = df_recibos_virtual['CLASE_ACTUAL'].str.strip()
df_recibos_virtual['ESTADO_PAGO'] = df_recibos_virtual['ESTADO_PAGO'].str.strip()
df_recibos_virtual['EST_MATRICULADO'] = (
    df_recibos_virtual['EST_MATRICULADO']
    .astype(str)
    .str.strip()
    .str.upper()
)

In [57]:
# se divide entre validos
df_validos_virtual = df_recibos_virtual[
    ((df_recibos_virtual['FuerzaComercialFinal'] == 'BASICA Y MEDIA')  | (df_recibos_virtual['FuerzaComercialFinal'] == 'VINCULACIONES') | (df_recibos_virtual['FuerzaComercialFinal'] == 'INDETERMINADO')) &
    (df_recibos_virtual['NUEVO'] == 'NUEVO') &
    (df_recibos_virtual['EST_MATRICULADO'] == '1-ACTIVO') & 
    (df_recibos_virtual['ESTADO_PAGO'] == 'PAGO')
].copy()
df_validos_virtual.drop_duplicates(inplace=True)
df_validos_virtual.shape

(277, 16)

In [58]:
# se divide entre no validos
mask = (
    df_recibos_virtual['FuerzaComercialFinal'].isin(
        ['BASICA Y MEDIA', 'VINCULACIONES', 'INDETERMINADO']
    ) &
    (df_recibos_virtual['NUEVO'] == 'NUEVO') &
    (df_recibos_virtual['EST_MATRICULADO'] == '1-ACTIVO') &
    (df_recibos_virtual['ESTADO_PAGO'] == 'PAGO')
)
df_no_validos_virtual = df_recibos_virtual[~mask].copy()
df_no_validos_virtual.drop_duplicates(inplace=True)
df_no_validos_virtual.shape

(34, 16)

In [59]:
# Inicializar MOTIVO vacío
df_no_validos_virtual['MOTIVO'] = ''

# Condición 1
condicion = df_no_validos_virtual['ESTADO_PAGO'] == 'NO PAGO'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'NO REGISTRA PAGO | '

# Condición 2
condicion = (
    (df_no_validos_virtual['NUEVO'] != 'NUEVO')
)
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'NO ES UN ESTUDIANTE NUEVO | '

# Condición 3
condicion = df_no_validos_virtual['FuerzaComercialFinal'] != 'VINCULACIONES'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'PERTENECE A OTRA FUERZA | '

# Condición 4
condicion = df_no_validos_virtual['EST_MATRICULADO'] != '1-ACTIVO'
df_no_validos_virtual.loc[condicion, 'MOTIVO'] += 'MATRICULA INACTIVA | '

# Quitar el último " | "
df_no_validos_virtual['MOTIVO'] = (
    df_no_validos_virtual['MOTIVO']
    .str.rstrip(' |')
)

# Los que no tengan ningún motivo quedan como N/A
df_no_validos_virtual.loc[
    df_no_validos_virtual['MOTIVO'] == '',
    'MOTIVO'
] = 'N/A'

df_no_validos_virtual

,DOC_ALUM,PERIODO,ESTADO_PAGO,FuerzaComercialFinal,NUEVO,CLASE_ACTUAL,TIPO,PROGRAMA,EST_MATRICULADO,LLAVE_RECIBOS,Propietario del registro,Modalidad,Correo Ejecutivo,Regional,Documento Ejecutivo,Llave,MOTIVO
1,1000148866,26V05,PAGO,TELECAMPUS,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS CONTABLES,1-ACTIVO,1000148866,Vinculaciones Paula Alejandra Lopez Correa,Virtual,PAULA_LOPEZCO@CUN.EDU.CO,Bogotá,1053814476,1000148866,PERTENECE A OTRA FUERZA
2,1000217592,26V05,PAGO,CONTACT,NUEVO,NUEVO,Pregrado,DIRECCIÓN Y PRODUCCIÓN DE MEDIOS AUDIOVISUALES,1-ACTIVO,1000217592,Vinculaciones Laura Camila Castillo Sepulveda,Presencial,LAURA_CASTILLOSEP@CUN.EDU.CO,Bogotá,1125230726,1000217592,PERTENECE A OTRA FUERZA
3,1000351487,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS OPERATIVOS DE ...,1-ACTIVO,1000351487,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,1000351487,NO REGISTRA PAGO
7,1000789899,26V05,PAGO,PROYECTOS ESPECIALES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN REGISTRO DE PROCESOS PR...,1-ACTIVO,1000789899,Vinculaciones Erika Andrea Arias Molano,Virtual,ERIKA_ARIASM@CUN.EDU.CO,Bogotá,1024525114,1000789899,PERTENECE A OTRA FUERZA
25,1005777254,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PROCESOS CONTABLES,1-ACTIVO,1005777254,Vinculaciones Laura Camila Castillo Sepulveda,Virtual,LAURA_CASTILLOSEP@CUN.EDU.CO,Bogotá,1125230726,1005777254,NO REGISTRA PAGO
33,1007160052,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,1007160052,Vinculaciones Erika Andrea Ramirez Rico,Virtual,ERIKA_RAMIREZ@CUN.EDU.CO,Bogotá,1012382480,1007160052,NO REGISTRA PAGO
44,1010211957,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN PRODUCCIÓN DE MATERIAS ...,1-ACTIVO,1010211957,Vinculaciones Dairon Alejandro Ibañez Rey,Virtual,DAIRON_IBANEZR@CUN.EDU.CO,Bogotá,1019115249,1010211957,NO REGISTRA PAGO
47,1012370878,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,1012370878,Vinculaciones Katherine Acosta Rojas,Virtual,KATHERINE_ACOSTA@CUN.EDU.CO,Bogotá,1030576351,1012370878,NO REGISTRA PAGO
49,1012378352,26ES5,PAGO,PERMANENCIA,ANTIGUO,NUEVO,Postgrado,ESPECIALIZACIÓN EN GERENCIA EDUCATIVA,1-ACTIVO,1012378352,Vinculaciones Diego Alexander Gonzalez Arbelaez,Virtual,DIEGO_GONZALEZAR@CUN.EDU.CO,Bogotá,1012421425,1012378352,NO ES UN ESTUDIANTE NUEVO | PERTENECE A OTRA F...
54,1013117576,26V05,NO PAGO,VINCULACIONES,NUEVO,NUEVO,Pregrado,TÉCNICA PROFESIONAL EN SOPORTE DE SISTEMAS E I...,1-ACTIVO,1013117576,Vinculaciones Sara Valentina Castañeda Espitia,Virtual,SARA_CASTANEDAE@CUN.EDU.CO,Bogotá,1014187111,1013117576,NO REGISTRA PAGO


In [60]:
# se traen las ventas que no tienen marca de pago
df_no_recibos_virtual = df_ventas_virtual[~df_ventas_virtual['Llave'].isin(df_recibos_virtual['LLAVE_RECIBOS'])].copy()
df_no_recibos_virtual['DOC_ALUM'] = df_no_recibos_virtual['Número de Documento de estudiante'].astype(str)
df_no_recibos_virtual['PERIODO'] = df_no_recibos_virtual['Periodo']
df_no_recibos_virtual['ESTADO_PAGO'] = ' '
df_no_recibos_virtual['EST_MATRICULADO'] = ' '
df_no_recibos_virtual['FuerzaComercialFinal'] = ' '
df_no_recibos_virtual['NUEVO'] = ' '
df_no_recibos_virtual['CLASE_ACTUAL'] = ' '
df_no_recibos_virtual['TIPO'] = ' '
df_no_recibos_virtual['PROGRAMA'] = df_no_recibos_virtual['Programa vendido'] 
df_no_recibos_virtual['LLAVE_RECIBOS'] = ' '
df_no_recibos_virtual['MOTIVO'] = 'NO CUENTA CON MARCA DE PAGO PARA ESE PERIODO'
df_no_recibos_virtual = df_no_recibos_virtual[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO', 'LLAVE_RECIBOS',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo', 'Llave','MOTIVO']]
df_no_recibos_virtual.shape

(24, 17)

In [61]:
# SE UNIFICAN LOS NO VALIDOS Y LOS QUE NO TIENEN MARCA DE PAGO
df_no_validos_2 = pd.concat([df_no_validos_virtual,df_no_recibos_virtual])
df_no_validos_2.shape

(58, 17)

SE TRAEN LAS OPORTUNIDADES DE ZOHO CRM

In [62]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_zoho = pd.read_sql(f'''
Select distinct 
o.Propietario_de_Oportunidad_Name,
o.Hora_de_creación,
o.Fecha_de_cierre,
o.Fase,
o.Hora_de_modificación,
o.Número_de_Documento,
o.Programa_de_interés,
o.PERIODO,
o.Hora_de_la_última_actividad,
u.Correo_electrónico
from zoho.CRM.Oportunidades o
left join zoho.CRM.Usuarios U
ON o.Propietario_de_Oportunidad = u.Id
    where Periodo in ({periodos_sql})
and Fase LIKE '%Matriculado%'
and o.Workdrive_Folder_ID is not null
                               ''', engine2_1)
    Consulta_z2 = df_zoho.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_zoho.shape

Conexion a la base de datos realizada


Conexión cerrada


(14978, 10)

In [63]:
df_zoho['Correo_electrónico'] = df_zoho['Correo_electrónico'].str.upper()
df_zoho['LLAVE_ZOHO']=df_zoho['Número_de_Documento'].astype(str)+df_zoho['PERIODO'].astype(str).str.strip()+df_zoho['Correo_electrónico']
df_zoho_vinculaciones = df_zoho[df_zoho['Correo_electrónico'].isin(df_pre_ventas['Correo Ejecutivo'])].copy()
df_zoho_vinculaciones = df_zoho_vinculaciones.drop_duplicates()
df_zoho_vinculaciones.shape

(2139, 11)

In [64]:
# se crea la llave para cruzar por doc alum+periodo+correo ejecutivo
df_validos_virtual['LLAVE_ZOHO'] = df_validos_virtual['DOC_ALUM']+df_validos_virtual['PERIODO']+df_validos_virtual['Correo Ejecutivo']

In [65]:
df_zoho_recibos_virtual = df_validos_virtual.merge(df_zoho_vinculaciones[['Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación','LLAVE_ZOHO']],left_on='LLAVE_ZOHO',right_on='LLAVE_ZOHO',how='inner').copy()
df_zoho_recibos_virtual.drop_duplicates(inplace=True)
df_zoho_recibos_virtual.shape

(227, 20)

In [66]:
df_no_validos_zoho = df_validos_virtual[~df_validos_virtual['LLAVE_ZOHO'].isin(df_zoho_vinculaciones['LLAVE_ZOHO'])].copy()
df_no_validos_zoho['MOTIVO'] = 'NO ESTA OK EN ZOHO'
df_no_validos_zoho = df_no_validos_zoho.drop(columns=['LLAVE_ZOHO'])
df_no_validos_zoho.shape

(52, 17)

In [67]:
# SE UNIFICAN LOS NO VALIDOS TOTALES DE VIRTUALES
df_no_validos_3 = pd.concat([df_no_validos_2,df_no_validos_zoho])
df_no_validos_3.shape

(110, 17)

# SE VALIDA QUE NO SE HAYA PAGADO 

In [68]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 
            select * from CUN_REPOSITORIO.AARBOLEDA.COMISIONES_VINCULACIONES_HIST
            ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_historico = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_historico.shape

(3571, 18)

In [69]:
df_historico['LLAVE_HISTORICO'] = df_historico['DOC_ALUM'].astype(str) + df_historico['PERIODO'].astype(str)

In [70]:
#------------------------------PROVISIONAL ASI DEBERIA SER LA  LLAVE DE T00000D0 EL PROCESO-----------------------
df_zoho_recibos_virtual['Llave'] = df_zoho_recibos_virtual['DOC_ALUM'].astype(str) + df_zoho_recibos_virtual['PERIODO'].astype(str)

In [71]:
df_validos_virtuales_final = df_zoho_recibos_virtual[~df_zoho_recibos_virtual['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_validos_virtuales_final.shape

(208, 20)

In [72]:
df_no_validos_historico = df_zoho_recibos_virtual[df_zoho_recibos_virtual['Llave'].isin(df_historico['LLAVE_HISTORICO'])].copy()
df_no_validos_historico['MOTIVO'] = 'YA SE CONTO EN LA LIQUIDACION' 
df_no_validos_historico = df_no_validos_historico.drop(columns=['LLAVE_ZOHO','Propietario_de_Oportunidad_Name','Fase','Hora_de_modificación'])
df_no_validos_historico.shape

(19, 17)

In [73]:
df_novalidos_virtual_final = pd.concat([df_no_validos_3,df_no_validos_historico])
df_novalidos_virtual_final.shape

(129, 17)

# SE TRAE EL HISTORICO DE VENTAS PARA REALIZAR LOS CALCULOS

In [74]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
            select * from CUN_REPOSITORIO.AARBOLEDA.BASEPAGO_VINCULACIONES_HIST
                         WHERE PERIODO_PAGO = ({periodo_pago})
                         AND LIDER LIKE '%LUZ%'
            ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_historico_basepago = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_historico_basepago.shape

(31, 24)

In [75]:
pre_ventas_asesor_virtuales = (
    df_validos_virtuales_final[df_validos_virtuales_final['PERIODO'] == periodo_virtual]
    .groupby('Documento Ejecutivo')
    .size()
    .reset_index(name='PRE_VENTAS_VIRTUALES')
)
pre_ventas_asesor_virtuales.shape

(15, 2)

In [76]:
pre_ventas_asesor_virtuales['Documento Ejecutivo'] = pre_ventas_asesor_virtuales['Documento Ejecutivo'].astype(str)
df_historico_basepago['Documento Ejecutivo'] = df_historico_basepago['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_virtuales = pre_ventas_asesor_virtuales.merge(df_historico_basepago[['Documento Ejecutivo','VENTAS_VIRTUALES']],how='right',on='Documento Ejecutivo')
pre_ventas_asesor_virtuales['PRE_VENTAS_VIRTUALES'] = (
    pre_ventas_asesor_virtuales['PRE_VENTAS_VIRTUALES'].fillna(0)
)

pre_ventas_asesor_virtuales['VENTAS_VIRTUALES'] = (
    pre_ventas_asesor_virtuales['VENTAS_VIRTUALES'].fillna(0)
)

pre_ventas_asesor_virtuales['TOTAL_VENTAS'] = (
    pre_ventas_asesor_virtuales['PRE_VENTAS_VIRTUALES'] +
    pre_ventas_asesor_virtuales['VENTAS_VIRTUALES']
)


In [77]:
pre_ventas_asesor_virtuales['Documento Ejecutivo'] = pre_ventas_asesor_virtuales['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_virtuales = pre_ventas_asesor_virtuales.drop(columns=['PRE_VENTAS_VIRTUALES','VENTAS_VIRTUALES'])
pre_ventas_asesor_virtuales = pre_ventas_asesor_virtuales.rename(columns={'TOTAL_VENTAS':'VENTAS_VIRTUALES'})

In [78]:
df_metas_virtual['GESTOR DOCUMENTO'] = df_metas_virtual['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_virtuales = pre_ventas_asesor_virtuales.merge(df_metas_virtual,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_virtuales = ventas_asesor_virtuales.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_virtuales

,Documento Ejecutivo,VENTAS_VIRTUALES,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD
0,1000335802,7.0,ANGIE_DIAZJ@CUN.EDU.CO,ESMERALDA,50,LUZ,BOGOTA,GRANDES CUENTAS
1,1000457601,14.0,DIEGO_BOHORQUEZCO@CUN.EDU.CO,PLATA,30,LUZ,BOGOTA,ENGATIVA
2,1001956960,5.0,PAULA_HURTADOM@CUN.EDU.CO,BRONCE,20,LUZ,BOGOTA,KENNEDY
3,1003068849,38.0,PAULA_VENCE@CUN.EDU.CO,DIAMANTE,40,LUZ,BOGOTA,RAFAEL URIBE URIBE
4,1007101612,44.0,DANIEL_PRIETO@CUN.EDU.CO,DIAMANTE,40,LUZ,BOGOTA,RAFAEL URIBE URIBE
5,1007702093,10.0,ANGGY_AGRAYVI@CUN.EDU.CO,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE
6,1007812062,1.0,JUAN_CASTANEDAA@CUN.EDU.CO,ESMERALDA,50,LUZ,BOGOTA,GRANDES CUENTAS
7,1012388460,56.0,DAIRON_TELLEZZA@CUN.EDU.CO,PLATA,30,LUZ,BOGOTA,BOSA
8,1012421425,25.0,DIEGO_GONZALEZAR@CUN.EDU.CO,BRONCE,20,LUZ,BOGOTA,SOACHA
9,1013099350,1.0,LEIDY_RINCON@CUN.EDU.CO,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE


# SE DEFINE EL ESQUEMA COMISIONAL

In [79]:
esquema_comisiones_virtuales = {
    "LUZ": {
        "ORO": {
                        60: [  
                                (60, float('inf'), 80000)
                            ]},

        "ESMERALDA": {
                        50: [  
                                (50, 59, 70000),
                                (60, float('inf'), 80000)
                            ]},
        "DIAMANTE": {
                        40: [  
                                (40, 49, 60000),
                                (50, 59, 70000),
                                (60, float('inf'), 80000)
                            ]},
        "PLATA": {
            
                        30: [  
                                (30, 39, 50000),    
                                (40, 49, 60000),
                                (50, 59, 70000),
                                (60, float('inf'), 80000)
                            ]},
        "BRONCE": {
            
                        20: [  
                                (20, 29, 40000),
                                (30, 39, 50000),    
                                (40, 49, 60000),
                                (50, 59, 70000),
                                (60, float('inf'), 80000)
                            ]},
        "HIERRO": {
            
                        10: [  
                                (10, 29, 40000),    
                                (30, 39, 50000),
                                (40, 49, 60000),
                                (50, 59, 70000),
                                (60, float('inf'), 80000)
                            ]}                               
    }}

In [80]:
def asignar_comision_virtuales(lider, categoria, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_virtuales:
        return 0

    # 2️⃣ Validar sede
    if categoria not in esquema_comisiones_virtuales[lider]:
        return 0

    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_virtuales[lider][categoria]:
        return 0

    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_virtuales[lider][categoria][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [81]:
ventas_asesor_virtuales["COMISION_TOTAL_VIRTUAL"] = ventas_asesor_virtuales.apply(
    lambda row: asignar_comision_virtuales(
        row["LIDER"],
        row["CATEGORIA"],
        row["Meta"],
        row["VENTAS_VIRTUALES"]
    ),
    axis=1
)
ventas_asesor_virtuales['COMISION_UNITARIA_VIRTUAL'] = ventas_asesor_virtuales['COMISION_TOTAL_VIRTUAL'] / ventas_asesor_virtuales['VENTAS_VIRTUALES']
ventas_asesor_virtuales['%CUMPLIMIENTO_VIRTUAL'] = (
    (ventas_asesor_virtuales['VENTAS_VIRTUALES'] / ventas_asesor_virtuales['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_virtuales

,Documento Ejecutivo,VENTAS_VIRTUALES,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_VIRTUAL,COMISION_UNITARIA_VIRTUAL,%CUMPLIMIENTO_VIRTUAL
0,1000335802,7.0,ANGIE_DIAZJ@CUN.EDU.CO,ESMERALDA,50,LUZ,BOGOTA,GRANDES CUENTAS,0.0,0.0,14.0%
1,1000457601,14.0,DIEGO_BOHORQUEZCO@CUN.EDU.CO,PLATA,30,LUZ,BOGOTA,ENGATIVA,0.0,0.0,46.67%
2,1001956960,5.0,PAULA_HURTADOM@CUN.EDU.CO,BRONCE,20,LUZ,BOGOTA,KENNEDY,0.0,0.0,25.0%
3,1003068849,38.0,PAULA_VENCE@CUN.EDU.CO,DIAMANTE,40,LUZ,BOGOTA,RAFAEL URIBE URIBE,0.0,0.0,95.0%
4,1007101612,44.0,DANIEL_PRIETO@CUN.EDU.CO,DIAMANTE,40,LUZ,BOGOTA,RAFAEL URIBE URIBE,2640000.0,60000.0,110.0%
5,1007702093,10.0,ANGGY_AGRAYVI@CUN.EDU.CO,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE,400000.0,40000.0,100.0%
6,1007812062,1.0,JUAN_CASTANEDAA@CUN.EDU.CO,ESMERALDA,50,LUZ,BOGOTA,GRANDES CUENTAS,0.0,0.0,2.0%
7,1012388460,56.0,DAIRON_TELLEZZA@CUN.EDU.CO,PLATA,30,LUZ,BOGOTA,BOSA,3920000.0,70000.0,186.67%
8,1012421425,25.0,DIEGO_GONZALEZAR@CUN.EDU.CO,BRONCE,20,LUZ,BOGOTA,SOACHA,1000000.0,40000.0,125.0%
9,1013099350,1.0,LEIDY_RINCON@CUN.EDU.CO,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE,0.0,0.0,10.0%


# PROCESO DE COMISION PRESENCIALES

In [82]:
pre_ventas_asesor_presenciales = (
    df_validos_virtuales_final[df_validos_virtuales_final['PERIODO'] == periodo_presencial]
    .groupby('Documento Ejecutivo')
    .size()
    .reset_index(name='PRE_VENTAS_PRESENCIALES')
)
pre_ventas_asesor_presenciales.shape

(3, 2)

In [83]:
pre_ventas_asesor_presenciales['Documento Ejecutivo'] = pre_ventas_asesor_presenciales['Documento Ejecutivo'].astype(str)
df_historico_basepago['Documento Ejecutivo'] = df_historico_basepago['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_presenciales = pre_ventas_asesor_presenciales.merge(df_historico_basepago[['Documento Ejecutivo','VENTAS_PRESENCIALES']],how='right',on='Documento Ejecutivo')
pre_ventas_asesor_presenciales['VENTAS_PRESENCIALES'] = (pre_ventas_asesor_presenciales['VENTAS_PRESENCIALES'].fillna(0))
pre_ventas_asesor_presenciales['PRE_VENTAS_PRESENCIALES'] = (pre_ventas_asesor_presenciales['PRE_VENTAS_PRESENCIALES'].fillna(0))
pre_ventas_asesor_presenciales['TOTAL_VENTAS'] = (pre_ventas_asesor_presenciales['PRE_VENTAS_PRESENCIALES'] + pre_ventas_asesor_presenciales['VENTAS_PRESENCIALES'])
pre_ventas_asesor_presenciales.shape

(31, 4)

In [84]:
pre_ventas_asesor_presenciales['Documento Ejecutivo'] = pre_ventas_asesor_presenciales['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_presenciales = pre_ventas_asesor_presenciales.drop(columns=['PRE_VENTAS_PRESENCIALES','VENTAS_PRESENCIALES'])
pre_ventas_asesor_presenciales = pre_ventas_asesor_presenciales.rename(columns={'TOTAL_VENTAS':'VENTAS_PRESENCIALES'})

In [85]:
df_metas_presencial['GESTOR DOCUMENTO'] = df_metas_presencial['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_presenciales = pre_ventas_asesor_presenciales.merge(df_metas_presencial,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_presenciales = ventas_asesor_presenciales.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_presenciales

,Documento Ejecutivo,VENTAS_PRESENCIALES,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD
0,1000335802,3.0,angie_diazj@cun.edu.co,ESMERALDA,20,LUZ,BOGOTA,GRANDES CUENTAS
1,1000457601,0.0,diego_bohorquezco@cun.edu.co,PLATA,10,LUZ,BOGOTA,ENGATIVA
2,1001956960,2.0,paula_hurtadom@cun.edu.co,BRONCE,5,LUZ,BOGOTA,KENNEDY
3,1003068849,3.0,paula_vence@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE
4,1007101612,9.0,daniel_prieto@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE
5,1007702093,0.0,anggy_agrayvi@cun.edu.co,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE
6,1007812062,0.0,juan_castanedaa@cun.edu.co,ESMERALDA,20,LUZ,BOGOTA,GRANDES CUENTAS
7,1012388460,1.0,dairon_tellezza@cun.edu.co,PLATA,10,LUZ,BOGOTA,BOSA
8,1012421425,0.0,diego_gonzalezar@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA
9,1013099350,0.0,leidy_rincon@cun.edu.co,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE


In [86]:
esquema_comisiones_presenciales = {

    "LUZ": {
        "ORO": {
                        25: [  
                                (25, float('inf'), 80000)
                            ]},

        "ESMERALDA": {
                        20: [  
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "DIAMANTE": {
                        15: [  
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "PLATA": {
            
                        10: [  
                                (10, 14, 50000),    
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "BRONCE": {
            
                        5: [  
                                (5, 9, 40000),
                                (10, 14, 50000),    
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]}           
    }}

In [87]:
def asignar_comision_presenciales(lider, categoria, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_presenciales:
        return 0

    # 2️⃣ Validar sede
    if categoria not in esquema_comisiones_presenciales[lider]:
        return 0
    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_presenciales[lider][categoria]:
        return 0
    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_presenciales[lider][categoria][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [88]:
ventas_asesor_presenciales["COMISION_TOTAL_PRESENCIAL"] = ventas_asesor_presenciales.apply(
    lambda row: asignar_comision_presenciales(
        row["LIDER"],
        row["CATEGORIA"],
        row["Meta"],
        row["VENTAS_PRESENCIALES"]
    ),
    axis=1
)
ventas_asesor_presenciales['COMISION_UNITARIA_PRESENCIAL'] = ventas_asesor_presenciales['COMISION_TOTAL_PRESENCIAL'] / ventas_asesor_presenciales['VENTAS_PRESENCIALES']
ventas_asesor_presenciales['%CUMPLIMIENTO_PRESENCIAL'] = (
    (ventas_asesor_presenciales['VENTAS_PRESENCIALES'] / ventas_asesor_presenciales['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_presenciales

,Documento Ejecutivo,VENTAS_PRESENCIALES,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_PRESENCIAL,COMISION_UNITARIA_PRESENCIAL,%CUMPLIMIENTO_PRESENCIAL
0,1000335802,3.0,angie_diazj@cun.edu.co,ESMERALDA,20,LUZ,BOGOTA,GRANDES CUENTAS,0.0,0.0,15.0%
1,1000457601,0.0,diego_bohorquezco@cun.edu.co,PLATA,10,LUZ,BOGOTA,ENGATIVA,0.0,NaN,0.0%
2,1001956960,2.0,paula_hurtadom@cun.edu.co,BRONCE,5,LUZ,BOGOTA,KENNEDY,0.0,0.0,40.0%
3,1003068849,3.0,paula_vence@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE,0.0,0.0,20.0%
4,1007101612,9.0,daniel_prieto@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE,0.0,0.0,60.0%
5,1007702093,0.0,anggy_agrayvi@cun.edu.co,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE,0.0,NaN,0.0%
6,1007812062,0.0,juan_castanedaa@cun.edu.co,ESMERALDA,20,LUZ,BOGOTA,GRANDES CUENTAS,0.0,NaN,0.0%
7,1012388460,1.0,dairon_tellezza@cun.edu.co,PLATA,10,LUZ,BOGOTA,BOSA,0.0,0.0,10.0%
8,1012421425,0.0,diego_gonzalezar@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA,0.0,NaN,0.0%
9,1013099350,0.0,leidy_rincon@cun.edu.co,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE,0.0,NaN,0.0%


#PROCESO DE LIQUIDACION DE ESPECIALIZACIONES

In [89]:
pre_ventas_asesor_especializaciones = (
    df_validos_virtuales_final[df_validos_virtuales_final['PERIODO'] == periodo_especializacion]
    .groupby('Documento Ejecutivo')
    .size()
    .reset_index(name='PRE_VENTAS_ESPECIALIZACION')
)
pre_ventas_asesor_especializaciones.shape

(11, 2)

In [90]:
pre_ventas_asesor_especializaciones['Documento Ejecutivo'] = pre_ventas_asesor_especializaciones['Documento Ejecutivo'].astype(str)
df_historico_basepago['Documento Ejecutivo'] = df_historico_basepago['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_especializaciones = pre_ventas_asesor_especializaciones.merge(df_historico_basepago[['Documento Ejecutivo','VENTAS_ESPECIALIZACION']],how='right',on='Documento Ejecutivo')
pre_ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION'] = (pre_ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION'].fillna(0))
pre_ventas_asesor_especializaciones['PRE_VENTAS_ESPECIALIZACION'] = (pre_ventas_asesor_especializaciones['PRE_VENTAS_ESPECIALIZACION'].fillna(0))
pre_ventas_asesor_especializaciones['TOTAL_VENTAS'] = (pre_ventas_asesor_especializaciones['PRE_VENTAS_ESPECIALIZACION'] + pre_ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION'])
pre_ventas_asesor_especializaciones.shape

(31, 4)

In [91]:
pre_ventas_asesor_especializaciones['Documento Ejecutivo'] = pre_ventas_asesor_especializaciones['Documento Ejecutivo'].astype(str)
pre_ventas_asesor_especializaciones = pre_ventas_asesor_especializaciones.drop(columns=['PRE_VENTAS_ESPECIALIZACION','VENTAS_ESPECIALIZACION'])
pre_ventas_asesor_especializaciones = pre_ventas_asesor_especializaciones.rename(columns={'TOTAL_VENTAS':'VENTAS_ESPECIALIZACION'})

In [92]:
df_metas_especializacion['GESTOR DOCUMENTO'] = df_metas_especializacion['GESTOR DOCUMENTO'].astype(str)
ventas_asesor_especializaciones = pre_ventas_asesor_especializaciones.merge(df_metas_especializacion,how='left',left_on='Documento Ejecutivo',right_on='GESTOR DOCUMENTO').copy()
ventas_asesor_especializaciones = ventas_asesor_especializaciones.drop(columns=['GESTOR DOCUMENTO'])
ventas_asesor_especializaciones

,Documento Ejecutivo,VENTAS_ESPECIALIZACION,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD
0,1000335802,6.0,angie_diazj@cun.edu.co,ESMERALDA,20,LUZ,BOGOTA,GRANDES CUENTAS
1,1000457601,9.0,diego_bohorquezco@cun.edu.co,PLATA,10,LUZ,BOGOTA,ENGATIVA
2,1001956960,12.0,paula_hurtadom@cun.edu.co,BRONCE,5,LUZ,BOGOTA,KENNEDY
3,1003068849,16.0,paula_vence@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE
4,1007101612,85.0,daniel_prieto@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE
5,1007702093,14.0,anggy_agrayvi@cun.edu.co,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE
6,1007812062,0.0,juan_castanedaa@cun.edu.co,ESMERALDA,20,LUZ,BOGOTA,GRANDES CUENTAS
7,1012388460,65.0,dairon_tellezza@cun.edu.co,PLATA,10,LUZ,BOGOTA,BOSA
8,1012421425,78.0,diego_gonzalezar@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA
9,1013099350,1.0,leidy_rincon@cun.edu.co,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE


In [93]:
esquema_comisiones_especializaciones = {

    "LUZ": {
        "ORO": {
                        25: [  
                                (25, float('inf'), 80000)
                            ]},

        "ESMERALDA": {
                        20: [  
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "DIAMANTE": {
                        15: [  
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "PLATA": {
            
                        10: [  
                                (10, 14, 50000),    
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "BRONCE": {
            
                        5: [  
                                (5, 9, 40000),
                                (10, 14, 50000),    
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]},
        "HIERRO": {
            
                        10: [  
                                (10, 14, 40000),    
                                (15, 19, 60000),
                                (20, 24, 70000),
                                (25, float('inf'), 80000)
                            ]}                               
    }}

In [94]:
def asignar_comision_especializaciones(lider, categoria, meta, ventas):

    # 1️⃣ Validar líder
    if lider not in esquema_comisiones_especializaciones:
        return 0

    # 2️⃣ Validar sede
    if categoria not in esquema_comisiones_especializaciones[lider]:
        return 0
    # 4️⃣ Validar meta
    if meta not in esquema_comisiones_especializaciones[lider][categoria]:
        return 0

    # 5️⃣ Validar cumplimiento de meta
    if ventas < meta:
        return 0

    # 6️⃣ Obtener los pisos correspondientes a la meta
    rangos = esquema_comisiones_especializaciones[lider][categoria][meta]

    # 7️⃣ Buscar el piso correspondiente según las ventas
    for min_rango, max_rango, comision in rangos:

        if min_rango <= ventas <= max_rango:

            # Comisión por venta × número de ventas
            return ventas * comision

    return 0

In [95]:
ventas_asesor_especializaciones["COMISION_TOTAL_ESPECIALIZACION"] = ventas_asesor_especializaciones.apply(
    lambda row: asignar_comision_especializaciones(
        row["LIDER"],
        row["CATEGORIA"],
        row["Meta"],
        row["VENTAS_ESPECIALIZACION"]
    ),
    axis=1
)
ventas_asesor_especializaciones['COMISION_UNITARIA_ESPECIALIZACION'] = ventas_asesor_especializaciones['COMISION_TOTAL_ESPECIALIZACION'] / ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION']
ventas_asesor_especializaciones['%CUMPLIMIENTO_ESPECIALIZACION'] = (
    (ventas_asesor_especializaciones['VENTAS_ESPECIALIZACION'] / ventas_asesor_especializaciones['Meta']) * 100).round(2).astype(str) + '%'
ventas_asesor_especializaciones

,Documento Ejecutivo,VENTAS_ESPECIALIZACION,GESTOR CORREO,CATEGORIA,Meta,LIDER,SEDE,LOCALIDAD,COMISION_TOTAL_ESPECIALIZACION,COMISION_UNITARIA_ESPECIALIZACION,%CUMPLIMIENTO_ESPECIALIZACION
0,1000335802,6.0,angie_diazj@cun.edu.co,ESMERALDA,20,LUZ,BOGOTA,GRANDES CUENTAS,0.0,0.0,30.0%
1,1000457601,9.0,diego_bohorquezco@cun.edu.co,PLATA,10,LUZ,BOGOTA,ENGATIVA,0.0,0.0,90.0%
2,1001956960,12.0,paula_hurtadom@cun.edu.co,BRONCE,5,LUZ,BOGOTA,KENNEDY,600000.0,50000.0,240.0%
3,1003068849,16.0,paula_vence@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE,960000.0,60000.0,106.67%
4,1007101612,85.0,daniel_prieto@cun.edu.co,DIAMANTE,15,LUZ,BOGOTA,RAFAEL URIBE URIBE,6800000.0,80000.0,566.67%
5,1007702093,14.0,anggy_agrayvi@cun.edu.co,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE,560000.0,40000.0,140.0%
6,1007812062,0.0,juan_castanedaa@cun.edu.co,ESMERALDA,20,LUZ,BOGOTA,GRANDES CUENTAS,0.0,NaN,0.0%
7,1012388460,65.0,dairon_tellezza@cun.edu.co,PLATA,10,LUZ,BOGOTA,BOSA,5200000.0,80000.0,650.0%
8,1012421425,78.0,diego_gonzalezar@cun.edu.co,BRONCE,5,LUZ,BOGOTA,SOACHA,6240000.0,80000.0,1560.0%
9,1013099350,1.0,leidy_rincon@cun.edu.co,HIERRO,10,LUZ,BOGOTA,SABANA OCCIDENTE,0.0,0.0,10.0%


# SE ARMA LOS DATAFRAME FINAL

In [96]:
ventas_asesor_virtuales = ventas_asesor_virtuales.rename(columns={'Meta':'META_VIRTUAL'})
ventas_asesor_presenciales = ventas_asesor_presenciales.rename(columns={'Meta':'META_PRESENCIAL'})
ventas_asesor_especializaciones = ventas_asesor_especializaciones.rename(columns={'Meta':'META_ESPECIALIZACION'})
base_pago = ventas_asesor_virtuales.merge(
    ventas_asesor_presenciales[
        [
            'Documento Ejecutivo',
            'VENTAS_PRESENCIALES',
            'META_PRESENCIAL',
            'COMISION_TOTAL_PRESENCIAL',
            'COMISION_UNITARIA_PRESENCIAL',
            '%CUMPLIMIENTO_PRESENCIAL'
        ]
    ],
    how='left',
    on='Documento Ejecutivo'
).copy()


In [97]:
base_pago_final = base_pago.merge(
    ventas_asesor_especializaciones[
        [
       'Documento Ejecutivo', 'VENTAS_ESPECIALIZACION','META_ESPECIALIZACION', 
       'COMISION_TOTAL_ESPECIALIZACION', 'COMISION_UNITARIA_ESPECIALIZACION',
       '%CUMPLIMIENTO_ESPECIALIZACION'
        ]
    ],
    how='left',
    on='Documento Ejecutivo'
).copy()

In [99]:
columnas_ventas = [
    'VENTAS_VIRTUALES',
    'VENTAS_PRESENCIALES',
    'VENTAS_ESPECIALIZACION'
]

columnas_comisiones = [
    'COMISION_TOTAL_VIRTUAL',
    'COMISION_TOTAL_PRESENCIAL',
    'COMISION_TOTAL_ESPECIALIZACION'
]

base_pago_final['VENTAS_TOTALES'] = (
    base_pago_final[columnas_ventas].fillna(0).sum(axis=1)
)

base_pago_final['COMISION_FINAL'] = (
    base_pago_final[columnas_comisiones].fillna(0).sum(axis=1)
)
base_pago_final['PERIODO_PAGO'] = periodo_pago
base_pago_final = base_pago_final[['Documento Ejecutivo','GESTOR CORREO', 'CATEGORIA', 'LIDER', 'SEDE', 'LOCALIDAD', 'PERIODO_PAGO','VENTAS_VIRTUALES', 
       'META_VIRTUAL','%CUMPLIMIENTO_VIRTUAL', 'COMISION_UNITARIA_VIRTUAL','COMISION_TOTAL_VIRTUAL',
       'VENTAS_PRESENCIALES', 'META_PRESENCIAL','%CUMPLIMIENTO_PRESENCIAL','COMISION_UNITARIA_PRESENCIAL','COMISION_TOTAL_PRESENCIAL',
       'VENTAS_ESPECIALIZACION', 'META_ESPECIALIZACION', '%CUMPLIMIENTO_ESPECIALIZACION', 'COMISION_UNITARIA_ESPECIALIZACION',
       'COMISION_TOTAL_ESPECIALIZACION','VENTAS_TOTALES', 'COMISION_FINAL']]

# SE ARMA EL ARCHIVO FINAL

In [100]:
df_historico = df_historico[df_historico['PERIODO_PAGO'].astype(str) == periodo_pago]
df_historico.shape

(3571, 19)

In [101]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
            select * from CUN_REPOSITORIO.AARBOLEDA.NOVALIDOS_VINCULACIONES_HIST
                         WHERE PERIODO_PAGO = ({periodo_pago})
            ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_historico_novalidos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_historico_novalidos.shape

(160, 16)

In [102]:
df_historico_novalidos_2 = df_historico_novalidos[~df_historico_novalidos['DOC_ALUM'].astype(str).isin(df_validos_virtuales_final['DOC_ALUM'])].copy()
df_historico_novalidos_2.shape

(160, 16)

In [103]:
df_novalidos_virtual_final = df_novalidos_virtual_final.drop(columns=['Llave'])
no_validos_total = pd.concat([df_historico_novalidos_2,df_novalidos_virtual_final])
no_validos_total['PERIODO_PAGO'] = periodo_pago
no_validos_total = no_validos_total[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo','Documento Ejecutivo', 'Regional',
       'PERIODO_PAGO', 'MOTIVO']]
no_validos_total.shape

(289, 16)

In [104]:
df_validos_virtuales_final['PERIODO_PAGO'] = periodo_pago
df_validos_virtuales_final = df_validos_virtuales_final[['DOC_ALUM', 'PERIODO', 'ESTADO_PAGO', 'FuerzaComercialFinal', 'NUEVO',
       'CLASE_ACTUAL', 'TIPO', 'PROGRAMA', 'EST_MATRICULADO',
       'Propietario del registro', 'Modalidad', 'Correo Ejecutivo', 'Regional',
       'Documento Ejecutivo','Propietario_de_Oportunidad_Name', 'Fase', 'Hora_de_modificación', 'PERIODO_PAGO']]

In [105]:
with pd.ExcelWriter(
    f"LIQUIDACION_VINCULACIONES_BOGOTA_{periodo_pago}_2.xlsx",
    engine="openpyxl"
) as writer:

    base_pago_final.to_excel(
        writer,
        sheet_name="Comisiones_Asesores",
        index=False
    )

    df_historico.to_excel(
        writer,
        sheet_name="Validos",
        index=False
    )
    df_validos_virtuales_final.to_excel(
        writer,
        sheet_name="Novedades_Validas",
        index=False
    )

    no_validos_total.to_excel(
        writer,
        sheet_name="No_Validas",
        index=False
    )